In [41]:
import json
import utils
import pandas as pd
from service import group_rows_by_contractor, build_contractor_context, compute_contractor_scores
from hsecore.utils.parsers import rows_from_dicts
from dotenv import load_dotenv

load_dotenv()

False

In [42]:
import aisuite as ai

client = ai.Client()

In [43]:
utils.create_risk_db()

Inserted 40 contractors.
Created 'risk_operations.db' with 120 monthly report rows across 3 months.


In [44]:
db_path = "risk_operations.db"
left_join_query = utils.build_contractors_monthly_left_join_query()

csv_like_rows = utils.fetch_csv_like_monthly_rows(db_path=db_path, query=left_join_query)
df = pd.DataFrame(csv_like_rows)
df.head()

,contractor_id,contractor,size_tier,industry,month,hours,operated,recordables,lti,hipo,actions_open,actions_closed,actions_overdue,critical_overdue,exec_walks,exec_crit_findings,monthly_close_submitted,rejected_reports,docs_blocked,docs_at_risk
0,30,Acme Industrial Solutions,large,marine,2026-01,2253,1,0,0,0,20,14,8,8,1,1,1,2,0,0
1,30,Acme Industrial Solutions,large,marine,2026-02,2099,1,0,0,0,8,1,2,2,3,3,1,2,0,0
2,30,Acme Industrial Solutions,large,marine,2026-03,2956,1,1,0,1,32,31,3,2,2,0,1,0,0,0
3,2,Apex Heavy Industries,medium,construction,2026-01,1421,1,0,0,0,21,16,4,3,2,3,1,1,0,0
4,2,Apex Heavy Industries,medium,construction,2026-02,1146,1,0,0,0,11,9,0,0,0,0,1,1,1,0


In [45]:
contractor_reports_dict = utils.fetch_csv_like_monthly_rows(db_path=db_path, query=left_join_query)
contractor_reports = rows_from_dicts(contractor_reports_dict)
len(contractor_reports_dict), len(contractor_reports), contractor_reports[:50]

(120,
 120,
 [Row(contractor='Acme Industrial Solutions', month='2026-01', hours=2253.0, operated=1, monthly_close_submitted=1, extra={'contractor_id': 30, 'size_tier': 'large', 'industry': 'marine', 'recordables': 0, 'lti': 0, 'hipo': 0, 'actions_open': 20, 'actions_closed': 14, 'actions_overdue': 8, 'critical_overdue': 8, 'exec_walks': 1, 'exec_crit_findings': 1, 'rejected_reports': 2, 'docs_blocked': 0, 'docs_at_risk': 0}),
  Row(contractor='Acme Industrial Solutions', month='2026-02', hours=2099.0, operated=1, monthly_close_submitted=1, extra={'contractor_id': 30, 'size_tier': 'large', 'industry': 'marine', 'recordables': 0, 'lti': 0, 'hipo': 0, 'actions_open': 8, 'actions_closed': 1, 'actions_overdue': 2, 'critical_overdue': 2, 'exec_walks': 3, 'exec_crit_findings': 3, 'rejected_reports': 2, 'docs_blocked': 0, 'docs_at_risk': 0}),
  Row(contractor='Acme Industrial Solutions', month='2026-03', hours=2956.0, operated=1, monthly_close_submitted=1, extra={'contractor_id': 30, 'size_ti

In [46]:
# Features adicionales configurables (vacíos por defecto)
# Descomentar y modificar según necesidad
risk_features = [
    # FeatureSpec(name="incident_report_delay_days", kind="delay_days", 
    #             weight=0.2, cap=30, direction="bad", when_missing="neutral"),
]
trust_features = [
    # FeatureSpec(name="last_login_days", kind="delay_days", 
    #             weight=0.3, cap=60, direction="bad", when_missing="penalize"),
    # FeatureSpec(name="active_weeks", kind="count", 
    #             weight=1.0, cap=4, direction="good", when_missing="neutral"),
]
explain = True  # Configurable: si se desea explicación de los scores
grouped_by_contractor = group_rows_by_contractor(contractor_reports).items()
for contractor_name, contractor_data in grouped_by_contractor:
    context = build_contractor_context(contractor_name, contractor_data)
    scores = compute_contractor_scores(
        context,
        risk_features=risk_features,
        trust_features=trust_features,
        explain=explain,
    )
    print(f"Contractor: {contractor_name}")
    print(f"Scores: {scores}")

Contractor: Acme Industrial Solutions
Scores: BaseScoreResult(gate_status='OK', risk_score=21.789162561576354, trust_score=75.0, stability_index=28.234039408866998, risk_contribs=[('event_rate_per_1000h', 7.389162561576355), ('critical_overdue', 7.0), ('actions_overdue', 2.4000000000000004), ('trend_worsening', 5.0)], trust_contribs=[('base', 70.0), ('consistent_closes_bonus', 5.0)], drivers_top3=['base ↑', 'event_rate_per_1000h ↑', 'critical_overdue ↑'])
Contractor: Apex Heavy Industries
Scores: BaseScoreResult(gate_status='OK', risk_score=4.3, trust_score=60.0, stability_index=20.365000000000002, risk_contribs=[('event_rate_per_1000h', 0.0), ('critical_overdue', 3.5), ('actions_overdue', 0.8)], trust_contribs=[('base', 70.0), ('rejected_reports', -15.0), ('consistent_closes_bonus', 5.0)], drivers_top3=['base ↑', 'rejected_reports ↓', 'consistent_closes_bonus ↑'])
Contractor: Atlas Mechanical Works
Scores: BaseScoreResult(gate_status='OK', risk_score=10.143429532081814, trust_score=75